## Block 0 — Smart Stratified Sampling (Run Before Everything)
Shortlists your full dataset to **~3,500 cases** while guaranteeing every `keyword_group` tag has **10–20 cases minimum**.

**How it works:**
1. First pass — collects minimum quota (10 cases) for every keyword that has fewer than 10 in the full data
2. Second pass — fills remaining slots up to 3,500 by sampling proportionally from all keywords
3. Rare keywords get their quota protected; dominant keywords get capped
4. Saves shortlisted file to `saved_data/shortlisted_3500.csv` — used as `DATA_FILE` in Block 2

> Set `RAW_FILE` to your original full CSV. Block 2's `DATA_FILE` is automatically updated.


In [ ]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path

# ── CONFIGURATION ─────────────────────────────────────────────
RAW_FILE        = 'cfpb_complaints_simulated.csv'   # <-- YOUR FULL RAW CSV
TARGET_N        = 3500    # total cases to keep
MIN_PER_KEYWORD = 10      # guaranteed minimum per keyword
MAX_PER_KEYWORD = 20      # soft cap for rare keywords (prevents over-quota)
RANDOM_SEED     = 42
Path('saved_data').mkdir(exist_ok=True)

# ── LOAD ──────────────────────────────────────────────────────
df_raw = pd.read_csv(RAW_FILE)
print(f"Raw file: {len(df_raw):,} rows  |  {df_raw.shape[1]} columns")

# ── PARSE keyword_group ────────────────────────────────────────
def parse_kw(val):
    if isinstance(val, list): return [str(x).strip() for x in val if str(x).strip()]
    if isinstance(val, str):
        val = val.strip()
        if val.startswith('['):
            try: return [str(x).strip() for x in ast.literal_eval(val) if str(x).strip()]
            except: pass
        return [x.strip() for x in val.split(',') if x.strip()]
    return []

df_raw['_kw_parsed'] = df_raw['keyword_group'].apply(parse_kw)

# ── BUILD KEYWORD → ROW INDEX MAP ────────────────────────────
from collections import defaultdict
kw_to_idx = defaultdict(list)
for i, kws in enumerate(df_raw['_kw_parsed']):
    for kw in kws:
        kw_to_idx[kw].append(i)

all_keywords = sorted(kw_to_idx.keys())
print(f"\nUnique keywords found: {len(all_keywords)}")
kw_counts = {kw: len(idxs) for kw, idxs in kw_to_idx.items()}
kw_series  = pd.Series(kw_counts).sort_values(ascending=False)
print(f"\nKeyword frequency (top 10):")
print(kw_series.head(10).to_string())
print(f"\nKeyword frequency (bottom 10):")
print(kw_series.tail(10).to_string())
print(f"\nKeywords with < {MIN_PER_KEYWORD} cases: {(kw_series < MIN_PER_KEYWORD).sum()}")
print(f"Keywords with < 20 cases:           {(kw_series < 20).sum()}")

# ── PHASE 1: PROTECT RARE KEYWORDS ────────────────────────────
# For every keyword, collect up to MAX_PER_KEYWORD guaranteed cases
rng          = np.random.RandomState(RANDOM_SEED)
selected_idx = set()
quota_log    = {}

for kw in all_keywords:
    idxs     = kw_to_idx[kw]
    n_need   = min(MAX_PER_KEYWORD, max(MIN_PER_KEYWORD, len(idxs)))
    # Prioritise rows not yet selected (fresh quota)
    fresh    = [i for i in idxs if i not in selected_idx]
    already  = [i for i in idxs if i in selected_idx]
    n_fresh  = min(n_need, len(fresh))
    chosen   = rng.choice(fresh, size=n_fresh, replace=False).tolist() if fresh else []
    selected_idx.update(chosen)
    quota_log[kw] = len(chosen) + len([i for i in already])

print(f"\nPhase 1 (guaranteed quota): {len(selected_idx):,} rows selected")

# ── PHASE 2: FILL TO TARGET_N ─────────────────────────────────
# Fill remaining slots by proportional sampling from all keywords
remaining_slots = TARGET_N - len(selected_idx)
unselected_idx  = [i for i in df_raw.index if i not in selected_idx]

if remaining_slots > 0 and unselected_idx:
    # Weight unselected rows by inverse keyword frequency (rare = higher weight)
    row_weights = np.zeros(len(unselected_idx))
    for j, idx in enumerate(unselected_idx):
        kws = df_raw.iloc[idx]['_kw_parsed']
        if kws:
            # Use inverse frequency so rows from rare keywords fill first
            row_weights[j] = np.mean([1.0 / max(kw_counts.get(kw, 1), 1) for kw in kws])
        else:
            row_weights[j] = 1.0 / len(df_raw)  # uniform fallback

    row_weights = row_weights / row_weights.sum()   # normalise to probabilities
    n_to_sample = min(remaining_slots, len(unselected_idx))
    extra_idx   = rng.choice(unselected_idx, size=n_to_sample, replace=False, p=row_weights)
    selected_idx.update(extra_idx.tolist())
    print(f"Phase 2 (fill to {TARGET_N:,}): added {len(extra_idx):,} rows")

# ── BUILD FINAL SHORTLISTED DF ────────────────────────────────
df_short = df_raw.iloc[sorted(selected_idx)].drop(columns=['_kw_parsed']).reset_index(drop=True)
print(f"\nFinal shortlisted size: {len(df_short):,} rows")

# ── VERIFY KEYWORD COVERAGE ───────────────────────────────────
df_short['_kw_parsed'] = df_short['keyword_group'].apply(parse_kw)
kw_counts_short = defaultdict(int)
for kws in df_short['_kw_parsed']:
    for kw in kws:
        kw_counts_short[kw] += 1

coverage_df = pd.DataFrame({
    'keyword':       list(kw_counts_short.keys()),
    'count_full':    [kw_counts.get(k, 0) for k in kw_counts_short],
    'count_short':   list(kw_counts_short.values()),
}).sort_values('count_short', ascending=False)

below_min = coverage_df[coverage_df['count_short'] < MIN_PER_KEYWORD]

print(f"\nKeyword coverage in shortlisted set:")
print(coverage_df.to_string(index=False))
print(f"\nKeywords below {MIN_PER_KEYWORD} cases: {len(below_min)}", end="")
if len(below_min):
    print(f" (these had < {MIN_PER_KEYWORD} total in the full dataset — cannot fix):")
    print(below_min[['keyword','count_full','count_short']].to_string(index=False))
else:
    print("  -- All keywords meet minimum quota!")

# ── SAVE ──────────────────────────────────────────────────────
df_short = df_short.drop(columns=['_kw_parsed'])
out_path = 'saved_data/shortlisted_3500.csv'
df_short.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"Set DATA_FILE = '{out_path}' in Block 2 to use this file")

# Auto-update DATA_FILE for the session
DATA_FILE = out_path
print(f"\nDATA_FILE updated to: {DATA_FILE}")


# Dynamic Complaint Tagger — Definitive Pipeline v4

**Run each block independently OR top-to-bottom as a full pipeline.**

| Block | What it does |
|---|---|
| 1 | Installs & imports |
| 2 | Load & parse data |
| 3 | Auto-diagnose & compute all settings |
| 4 | Drop rare tags |
| 5 | CleanLab noise detection (fast concatenated column method) |
| 6 | Train / Val / Test split + save |
| 7 | Imbalance handling (augment + undersample + weighted sampler) |
| 8 | Build input text + Dataset + DataLoaders |
| 9 | Model + Loss + Optimizer |
| 10 | Training loop with epoch checkpoints |
| 11 | Per-tag threshold tuning |
| 12 | Final test evaluation |
| 13 | Human review export |
| 14 | Feedback loop (after human review) |
| 15 | Inference on new complaints |

> **Only change needed:** Set `DATA_FILE` in Block 2 to your CSV filename.


## Block 1 — Installs & Imports

In [ ]:
import subprocess, sys
for pkg in ["transformers", "torch", "scikit-learn", "tqdm", "pandas", "numpy", "nbformat"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"])
try:
    import cleanlab
    print("cleanlab already installed")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "cleanlab", "-q"])
    print("cleanlab installed")


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import ast, random, os, json
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'distilbert-base-uncased'

for folder in ['checkpoints', 'saved_data', 'human_review']:
    Path(folder).mkdir(exist_ok=True)

# ── Checkpoint helpers used across all blocks ──────────────────
def checkpoint_exists(name):
    return Path(f'checkpoints/{name}').exists()

def save_checkpoint(obj, name):
    p = f'checkpoints/{name}'
    if isinstance(obj, pd.DataFrame): obj.to_csv(p, index=False)
    elif isinstance(obj, np.ndarray): np.save(p, obj)
    elif isinstance(obj, dict):
        with open(p, 'w') as f: json.dump(obj, f)
    print(f"  Saved: {name}")

def load_checkpoint(name):
    p = f'checkpoints/{name}'
    if name.endswith('.csv'):  return pd.read_csv(p)
    if name.endswith('.npy'):  return np.load(p, allow_pickle=True)
    if name.endswith('.json'):
        with open(p) as f: return json.load(f)

print(f"Device: {DEVICE}  |  Model: {MODEL_NAME}")
print("All imports OK — checkpoints/ saved_data/ human_review/ created")


## Block 2 — Load & Parse Data
> **Change `DATA_FILE` to your CSV filename before running.**

In [ ]:
DATA_FILE = 'cfpb_complaints_simulated.csv'   # <-- YOUR FILE HERE

def parse_tags(val):
    if isinstance(val, list): return [t.strip() for t in val if str(t).strip()]
    if isinstance(val, str):
        val = val.strip()
        if val.startswith('['):
            try: return [t.strip() for t in ast.literal_eval(val) if str(t).strip()]
            except: pass
        return [t.strip() for t in val.split(',') if t.strip()]
    return []

def parse_list_str(val):
    if isinstance(val, list): return val
    if isinstance(val, str) and val.startswith('['):
        try: return ast.literal_eval(val)
        except: pass
    return [val] if isinstance(val, str) and val.strip() else []

df = pd.read_csv(DATA_FILE)
df['tags_parsed']          = df['tier_2'].apply(parse_tags)
df['keyword_group_parsed'] = df['keyword_group'].apply(parse_list_str)
df = df[df['tags_parsed'].apply(len) > 0].reset_index(drop=True)

mlb = MultiLabelBinarizer()
y   = mlb.fit_transform(df['tags_parsed'])
tag_counts_s    = pd.Series(y.sum(axis=0), index=mlb.classes_).sort_values(ascending=False)
n_total         = len(df)
imbalance_ratio = tag_counts_s.max() / max(tag_counts_s.min(), 1)
tags_per_row    = y.sum(axis=1)

print(f"Loaded: {n_total} complaints  |  {len(mlb.classes_)} unique tier_2 tags")
print(f"Avg tags/complaint: {tags_per_row.mean():.2f}  |  Imbalance ratio: {imbalance_ratio:.0f}x")
print(f"\nTop 5 tags:\n{tag_counts_s.head().to_string()}")
print(f"\nBottom 5 tags:\n{tag_counts_s.tail().to_string()}")


## Block 3 — Auto-Diagnose & Compute All Settings
Reads your data and auto-computes every hyperparameter. Nothing hardcoded.

In [ ]:
HAS_CLEANLAB = 'cleanlab_avg_quality' in df.columns
if HAS_CLEANLAB:
    for col in ['has_label_issue', 'cleanlab_has_issue']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()                             .map({'TRUE':True,'FALSE':False,'1':True,'0':False}).fillna(False)
    avg_quality = df['cleanlab_avg_quality'].fillna(0.9).mean()
    pct_flagged = df['has_label_issue'].mean() if 'has_label_issue' in df.columns else 0.0
    df['noise_score'] = (1.0 - df['cleanlab_avg_quality'].fillna(avg_quality).clip(0,1))
else:
    avg_quality, pct_flagged = 0.9, 0.0
    tc = y.sum(axis=1); exp = tc.mean(); std = tc.std() + 1e-6
    df['noise_score'] = (np.abs(tc - exp) / (std * 3)).clip(0, 0.5)

MIN_TAG_SAMPLES = max(3, min(10, int(n_total * 0.005)))
RARE_THRESHOLD  = max(10, int(n_total * 0.03))
DOMINANT_FREQ   = 0.50
RARE_FREQ       = max(0.02, min(0.08, 1.5 / len(mlb.classes_)))
KW_DROPOUT      = min(0.55, 0.20 + (min(imbalance_ratio, 100) / 200))
BASE_SMOOTH     = round(min(0.15, max(0.02, 0.02 + (1.0 - avg_quality) * 0.25)), 3)
GAMMA_NEG       = int(min(6, max(2, 2 + min(imbalance_ratio, 100) / 30)))
EPOCHS          = max(5, min(12, int(20000 / n_total + 3)))
BATCH_SIZE      = 32 if n_total >= 5000 else 16 if n_total >= 1000 else 8
EARLY_STOP_PAT  = 3 if n_total >= 2000 else 4
LR              = 2e-5
med_chars       = df['summary'].dropna().str.len().median() if 'summary' in df.columns else 600
MAX_LEN         = 512 if med_chars > 1200 else 384 if med_chars > 600 else 256
DO_AUGMENT      = bool((tag_counts_s < RARE_THRESHOLD).any())
DO_UNDERSAMPLE  = bool(imbalance_ratio > 5 and (tag_counts_s / n_total > DOMINANT_FREQ).any())

print("DATA DIAGNOSIS REPORT")
print("=" * 55)
print(f"  Total complaints:       {n_total}")
print(f"  Unique tier_2 tags:     {len(mlb.classes_)}")
print(f"  Avg tags/complaint:     {tags_per_row.mean():.2f}")
print(f"  Imbalance ratio:        {imbalance_ratio:.0f}x")
print(f"  Most common tag:        '{tag_counts_s.index[0]}' ({int(tag_counts_s.max())} samples)")
print(f"  Least common tag:       '{tag_counts_s.index[-1]}' ({int(tag_counts_s.min())} samples)")
print(f"  Tags < 5 samples:       {(tag_counts_s < 5).sum()}")
print(f"  Tags < {RARE_THRESHOLD} samples:      {(tag_counts_s < RARE_THRESHOLD).sum()} (will be augmented)")
print(f"  CleanLab available:     {HAS_CLEANLAB}")
print(f"  Avg label quality:      {avg_quality:.3f}")
print(f"  Pct flagged noisy:      {pct_flagged*100:.1f}%")
print()
print("  AUTO-COMPUTED SETTINGS:")
print(f"  MIN_TAG_SAMPLES={MIN_TAG_SAMPLES}  RARE_THRESHOLD={RARE_THRESHOLD}  KW_DROPOUT={KW_DROPOUT:.2f}")
print(f"  BASE_SMOOTH={BASE_SMOOTH}  GAMMA_NEG={GAMMA_NEG}  EPOCHS={EPOCHS}")
print(f"  BATCH_SIZE={BATCH_SIZE}  MAX_LEN={MAX_LEN}  LR={LR}")
print(f"  DO_AUGMENT={DO_AUGMENT}  DO_UNDERSAMPLE={DO_UNDERSAMPLE}")
print("=" * 55)


## Block 4 — Drop Rare Tags
Removes tier_2 tags with fewer than `MIN_TAG_SAMPLES` examples — too few to learn from reliably.

In [ ]:
valid_mask = y.sum(axis=0) >= MIN_TAG_SAMPLES
dropped    = mlb.classes_[~valid_mask]
valid_tags = mlb.classes_[valid_mask]
y          = y[:, valid_mask]
NUM_LABELS = len(valid_tags)

print(f"Kept: {NUM_LABELS} tags  |  Dropped: {len(dropped)} tags with < {MIN_TAG_SAMPLES} samples")
if len(dropped): print(f"Dropped tags: {list(dropped)}")

keep = y.sum(axis=1) > 0
df, y = df[keep].reset_index(drop=True), y[keep]
print(f"Complaints after filter: {len(df)}")


## Block 5 — CleanLab Noise Detection (Fast Concatenated Column Method)
**The smart way:** one concatenated text column -> TF-IDF (capped at 15k features) -> cross-val probabilities -> CleanLab quality scores.

- Runs in ~2-3 minutes on 3000 rows
- Cached to `checkpoints/cleanlab_scores.csv` — never reruns unless you delete it
- Falls back to tag-count consistency scoring if CleanLab is not installed

In [ ]:
CLEANLAB_CACHE = 'checkpoints/cleanlab_scores.csv'

if HAS_CLEANLAB:
    print("CleanLab scores already in data — skipping computation")

elif checkpoint_exists('cleanlab_scores.csv'):
    print("Loading cached CleanLab scores...")
    cl_scores = load_checkpoint('cleanlab_scores.csv')
    df['cleanlab_avg_quality'] = cl_scores['cleanlab_avg_quality'].values
    df['cleanlab_has_issue']   = cl_scores['cleanlab_has_issue'].values
    df['noise_score']          = (1.0 - df['cleanlab_avg_quality'].clip(0,1))
    avg_quality = df['cleanlab_avg_quality'].mean()
    BASE_SMOOTH = round(min(0.15, max(0.02, 0.02 + (1.0 - avg_quality) * 0.25)), 3)
    print(f"  Avg quality: {avg_quality:.3f}  |  BASE_SMOOTH updated to: {BASE_SMOOTH}")

else:
    print("Running CleanLab — fast concatenated-column method...")

    # SMART WAY: single concatenated column — no separate columns fed to CleanLab
    df['cleanlab_input'] = (
        df['Service'].fillna('') + ' | ' +
        df['keywords_list'].astype(str).fillna('') + ' | ' +
        df['summary'].fillna('')   # summary last = richest signal
    )

    # Cap max_features=15000 -> halves run time vs default 50k+ with minimal quality loss
    tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,2), sublinear_tf=True)
    X_cl  = tfidf.fit_transform(df['cleanlab_input'])
    print(f"  TF-IDF matrix: {X_cl.shape}  (capped at 15k features for speed)")

    try:
        from cleanlab.multilabel_classification.label_quality_scores import (
            multilabel_label_quality_scores
        )
        clf_cl = OneVsRestClassifier(
            LogisticRegression(class_weight='balanced', max_iter=300, C=1.0), n_jobs=-1
        )
        print("  Computing 5-fold cross-val probabilities (approx 2 mins)...")
        pred_probs = cross_val_predict(clf_cl, X_cl, y, cv=5, method='predict_proba')
        quality    = multilabel_label_quality_scores(y, pred_probs)

        df['cleanlab_avg_quality'] = quality
        df['cleanlab_has_issue']   = quality < 0.7
        df['noise_score']          = (1.0 - quality.clip(0,1))

        cl_out = df[['case_reference','cleanlab_avg_quality','cleanlab_has_issue','noise_score']]
        cl_out.to_csv(CLEANLAB_CACHE, index=False)

        avg_quality = quality.mean()
        BASE_SMOOTH = round(min(0.15, max(0.02, 0.02 + (1.0 - avg_quality) * 0.25)), 3)

        print(f"  CleanLab done!")
        print(f"  Mean quality:           {quality.mean():.3f}")
        print(f"  Flagged noisy (< 0.7):  {(quality<0.7).sum()} ({(quality<0.7).mean()*100:.1f}%)")
        print(f"  Very noisy   (< 0.4):   {(quality<0.4).sum()} — consider dropping")
        print(f"  BASE_SMOOTH updated to: {BASE_SMOOTH}")
        print(f"  Cached to: {CLEANLAB_CACHE}")

    except ImportError:
        print("  cleanlab not installed — pip install cleanlab")
        print("  Using tag-count consistency as noise proxy instead")
        tc = y.sum(axis=1); exp = tc.mean(); std = tc.std() + 1e-6
        df['noise_score'] = (np.abs(tc - exp) / (std * 3)).clip(0, 0.5)

# Save cleaned dataframe after CleanLab
df.to_csv('saved_data/df_after_cleanlab.csv', index=False)
np.save('saved_data/y_labels.npy', y)
with open('saved_data/valid_tags.json','w') as f: json.dump(list(valid_tags), f)
print("Saved: saved_data/df_after_cleanlab.csv | y_labels.npy | valid_tags.json")


## Block 6 — Train / Val / Test Split
Split done **before** any augmentation to prevent data leakage. All splits saved to disk for reproducibility.

In [ ]:
idx = np.arange(len(df))
tr_idx, temp    = train_test_split(idx, test_size=0.2, random_state=42)
val_idx, te_idx = train_test_split(temp, test_size=0.5, random_state=42)

df_tr,  y_tr  = df.iloc[tr_idx].reset_index(drop=True), y[tr_idx]
df_val, y_val = df.iloc[val_idx].reset_index(drop=True), y[val_idx]
df_te,  y_te  = df.iloc[te_idx].reset_index(drop=True), y[te_idx]

df_tr.to_csv('saved_data/train_split.csv', index=False)
df_val.to_csv('saved_data/val_split.csv',  index=False)
df_te.to_csv('saved_data/test_split.csv',  index=False)
np.save('saved_data/y_tr.npy',  y_tr)
np.save('saved_data/y_val.npy', y_val)
np.save('saved_data/y_te.npy',  y_te)

print(f"Train: {len(df_tr)} | Val: {len(df_val)} | Test: {len(df_te)}")
print("Saved: train/val/test splits + label arrays to saved_data/")


## Block 7 — Imbalance Handling (3 Layers)
Applied in order on the **training set only**:
1. **Text augmentation** — copies of rare-tag complaints with minor word swaps/drops
2. **Undersampling** — removes 50% of rows that only have dominant tags
3. **WeightedRandomSampler** — rare-tag rows seen more per epoch

In [ ]:
# Layer 1: Text augmentation for rare tags
def augment_text(text):
    words = str(text).split()
    if len(words) < 6: return text
    op = random.choice(['swap', 'drop', 'repeat'])
    if op == 'swap' and len(words) > 3:
        i = random.randint(0, len(words)-2)
        words[i], words[i+1] = words[i+1], words[i]
    elif op == 'drop' and len(words) > 8:
        words.pop(random.randint(1, len(words)-2))
    elif op == 'repeat':
        mid = len(words)//2; words = words + words[mid:mid+6]
    return ' '.join(words)

if DO_AUGMENT:
    tag_counts_tr = y_tr.sum(axis=0)
    rare_idx      = np.where(tag_counts_tr < RARE_THRESHOLD)[0]
    aug_rows, aug_y = [], []
    for i in range(len(df_tr)):
        if y_tr[i, rare_idx].any():
            r = df_tr.iloc[i].copy()
            r['summary'] = augment_text(str(r.get('summary', '')))
            aug_rows.append(r); aug_y.append(y_tr[i])
    if aug_rows:
        df_tr = pd.concat([df_tr, pd.DataFrame(aug_rows)], ignore_index=True)
        y_tr  = np.vstack([y_tr, np.array(aug_y)])
        print(f"Layer 1: Augmented {len(aug_rows)} rare-tag rows -> train size: {len(df_tr)}")
else:
    print("Layer 1: Augmentation skipped (no rare tags below threshold)")

# Layer 2: Undersample dominant-only rows
if DO_UNDERSAMPLE:
    tag_freq_tr = y_tr.sum(axis=0) / len(y_tr)
    dom_set     = set(np.where(tag_freq_tr > DOMINANT_FREQ)[0])
    keep_i      = [i for i in range(len(df_tr))
                   if not set(np.where(y_tr[i]==1)[0]).issubset(dom_set)
                   or random.random() < 0.5]
    removed = len(df_tr) - len(keep_i)
    df_tr = df_tr.iloc[keep_i].reset_index(drop=True)
    y_tr  = y_tr[keep_i]
    print(f"Layer 2: Removed {removed} dominant-only rows -> train size: {len(df_tr)}")
else:
    print("Layer 2: Undersampling skipped (no dominant tags)")

# Layer 3: WeightedRandomSampler
tag_w = 1.0 / (y_tr.sum(axis=0) + 1e-6)
if 'sample_weight' in df_tr.columns:
    sw = df_tr['sample_weight'].fillna(1.0).values.astype(np.float32)
    sw = sw * (y_tr @ tag_w + 1e-6)
    print("Layer 3: Using existing sample_weight x rare-tag boost for sampler")
else:
    sw = (y_tr @ tag_w).astype(np.float32)
    print("Layer 3: Computed sample weights from inverse tag frequency")

sampler = WeightedRandomSampler(torch.tensor(sw, dtype=torch.float32), len(sw), replacement=True)
print(f"WeightedRandomSampler ready: {len(sw)} weights")


## Block 8 — Build Input Text, Dataset & DataLoaders
**Columns used:**
- `summary` — always (full complaint, richest signal)
- `Service` — always (product/service type)
- `tier2_consolidated_description` — always (semantic context)
- `keyword_group` — training: masked with `KW_DROPOUT` probability (forces model to read the complaint, not shortcut on keywords)

**Adaptive label smoothing:** noisy rows get more smoothing based on their `noise_score`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_text(row, is_training=False):
    parts = []
    for label, col in [('Service', 'Service'),
                        ('Complaint', 'summary'),
                        ('Context', 'tier2_consolidated_description')]:
        val = str(row.get(col, '')).strip()
        if val and val.lower() not in ('nan','none',''):
            parts.append(f"{label}: {val}")
    if not is_training or random.random() > KW_DROPOUT:
        kg = row.get('keyword_group_parsed', [])
        if isinstance(kg, list) and kg:
            parts.append(f"Categories: {', '.join(str(x) for x in kg)}")
    return ' [SEP] '.join(parts)

class ComplaintDataset(Dataset):
    def __init__(self, df_rows, labels, is_training=False):
        self.df = df_rows.reset_index(drop=True)
        self.labels = labels
        self.is_training = is_training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = tokenizer(build_text(row, self.is_training),
                        max_length=MAX_LEN, padding='max_length',
                        truncation=True, return_tensors='pt')
        lbl = torch.tensor(self.labels[idx], dtype=torch.float32)
        if self.is_training:
            ns     = float(row.get('noise_score', BASE_SMOOTH))
            smooth = min(0.20, max(0.01, BASE_SMOOTH + ns * 0.10))
            lbl    = lbl * (1 - smooth) + smooth * 0.5
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         lbl
        }

train_loader = DataLoader(ComplaintDataset(df_tr,  y_tr,  True),
                          batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader   = DataLoader(ComplaintDataset(df_val, y_val), batch_size=BATCH_SIZE, num_workers=0)
test_loader  = DataLoader(ComplaintDataset(df_te,  y_te),  batch_size=BATCH_SIZE, num_workers=0)

print(f"DataLoaders ready")
print(f"  Train: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}")
print(f"  MAX_LEN={MAX_LEN} | BATCH_SIZE={BATCH_SIZE} | KW_DROPOUT={KW_DROPOUT:.2f}")


## Block 9 — Model, Loss & Optimizer
- **DistilBERT** encoder with [CLS] pooling + dropout + linear head
- **Asymmetric Loss** with `gamma_neg` auto-scaled from imbalance ratio — penalises confident wrong predictions on dominant tags
- Swap `MODEL_NAME` to `bert-base-uncased` for slightly better performance if you have a GPU

In [ ]:
class ComplaintTagger(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout    = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, NUM_LABELS)

    def forward(self, input_ids, attention_mask):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(out.last_hidden_state[:, 0, :])   # [CLS] token
        return self.classifier(pooled)

class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05):
        super().__init__()
        self.gn, self.gp, self.clip = gamma_neg, gamma_pos, clip

    def forward(self, logits, targets):
        p  = torch.sigmoid(logits)
        pn = (1 - p + self.clip).clamp(max=1)
        return (-(targets    * torch.log(p.clamp(1e-8))  * (1-p)**self.gp
                + (1-targets)* torch.log(pn.clamp(1e-8)) * p**self.gn)).mean()

model     = ComplaintTagger().to(DEVICE)
criterion = AsymmetricLoss(gamma_neg=GAMMA_NEG)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, total_steps // 10),
    num_training_steps=total_steps
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {total_params:,} params  (66M = normal for DistilBERT)")
print(f"Loss: AsymmetricLoss(gamma_neg={GAMMA_NEG})  <- scaled from {imbalance_ratio:.0f}x imbalance")
print(f"Optimizer: AdamW | LR={LR} | Warmup={max(1, total_steps//10)} steps")


## Block 10 — Training Loop with Epoch Checkpoints
- Checkpoint saved after **every epoch** to `checkpoints/epoch_N.pt`
- Best model saved to `checkpoints/best_model.pt`
- Early stopping with patience auto-set by Block 3
- Training history saved to `saved_data/training_history.csv`

In [ ]:
def train_epoch(model, loader):
    model.train(); total = 0
    for b in tqdm(loader, desc="  Train", leave=False):
        optimizer.zero_grad()
        loss = criterion(
            model(b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE)),
            b['labels'].to(DEVICE))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total += loss.item()
    return total / len(loader)

def eval_epoch(model, loader):
    model.eval(); probs_all, labels_all = [], []
    with torch.no_grad():
        for b in tqdm(loader, desc="  Eval", leave=False):
            probs_all.append(
                torch.sigmoid(model(b['input_ids'].to(DEVICE),
                                    b['attention_mask'].to(DEVICE))).cpu().numpy())
            labels_all.append(b['labels'].numpy())
    return np.vstack(probs_all), np.vstack(labels_all)

print(f"Training: max {EPOCHS} epochs | patience={EARLY_STOP_PAT}\n")
best_f1, pat = 0.0, 0; history = []

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    tl     = train_epoch(model, train_loader)
    vp, vl = eval_epoch(model, val_loader)
    vl_h   = (vl >= 0.5).astype(int)
    macro  = f1_score(vl_h, (vp >= 0.5).astype(int), average='macro',  zero_division=0)
    micro  = f1_score(vl_h, (vp >= 0.5).astype(int), average='micro',  zero_division=0)
    history.append({'epoch': epoch+1, 'loss': round(tl,4),
                    'macro_f1': round(macro,4), 'micro_f1': round(micro,4)})
    print(f"  Loss: {tl:.4f}  |  Val Macro F1: {macro:.4f}  |  Val Micro F1: {micro:.4f}")

    # Save every epoch — lets you resume from any point
    torch.save({'epoch': epoch+1, 'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(), 'macro_f1': macro},
               f'checkpoints/epoch_{epoch+1}.pt')

    if macro > best_f1:
        best_f1 = macro
        torch.save(model.state_dict(), 'checkpoints/best_model.pt')
        print(f"  Best model saved (Macro F1: {best_f1:.4f})")
        pat = 0
    else:
        pat += 1
        if pat >= EARLY_STOP_PAT:
            print(f"  Early stop — best Macro F1: {best_f1:.4f}"); break

pd.DataFrame(history).to_csv('saved_data/training_history.csv', index=False)
print(f"\nTraining complete | Best Val Macro F1: {best_f1:.4f}")
print("Saved: checkpoints/best_model.pt | epoch_N.pt files | training_history.csv")


## Block 11 — Per-Tag Threshold Tuning
Finds the optimal decision threshold for each tag individually on the validation set.
- Dominant tags: threshold >= 0.65 (stop over-predicting)
- Rare tags: threshold <= 0.35 (catch more predictions)

In [ ]:
print("Loading best model and tuning per-tag thresholds...")
model.load_state_dict(torch.load('checkpoints/best_model.pt'))
vp, vl   = eval_epoch(model, val_loader)
vl_h     = (vl >= 0.5).astype(int)
tag_freq  = y_tr.sum(axis=0) / len(y_tr)
thresholds = []

for i in range(NUM_LABELS):
    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.10, 0.91, 0.05):
        f = f1_score(vl_h[:,i], (vp[:,i]>=t).astype(int), zero_division=0)
        if f > best_f: best_f, best_t = f, t
    if tag_freq[i] > DOMINANT_FREQ: best_t = max(best_t, 0.65)
    if tag_freq[i] < RARE_FREQ:     best_t = min(best_t, 0.35)
    thresholds.append(best_t)

thresholds = np.array(thresholds)
thr_df = pd.DataFrame({
    'tag': valid_tags,
    'train_freq_%': (tag_freq * 100).round(1),
    'threshold': thresholds,
    'category': ['dominant' if f > DOMINANT_FREQ
                 else 'rare' if f < RARE_FREQ
                 else 'normal' for f in tag_freq]
}).sort_values('train_freq_%', ascending=False)

np.save('saved_data/tag_thresholds.npy', thresholds)
thr_df.to_csv('saved_data/tag_info.csv', index=False)

print("Per-tag thresholds tuned and saved\n")
print(thr_df.to_string(index=False))


## Block 12 — Final Test Evaluation
Evaluates best model on the held-out test set using tuned thresholds.
**Macro F1 is your main metric** — treats all tags equally regardless of frequency.

In [ ]:
print("Running final test evaluation...")
tp, tl  = eval_epoch(model, test_loader)
tl_h    = (tl >= 0.5).astype(int)
preds   = (tp >= thresholds).astype(int)

macro_test = f1_score(tl_h, preds, average='macro',   zero_division=0)
micro_test = f1_score(tl_h, preds, average='micro',   zero_division=0)
samp_test  = f1_score(tl_h, preds, average='samples', zero_division=0)

print("=" * 55)
print(f"  Macro F1:    {macro_test:.4f}  <- main metric")
print(f"  Micro F1:    {micro_test:.4f}")
print(f"  Samples F1:  {samp_test:.4f}")
print("=" * 55)
print()
print(classification_report(tl_h, preds, target_names=valid_tags, zero_division=0))

pred_df = df_te[['case_reference']].copy()
for i, tag in enumerate(valid_tags):
    pred_df[f'pred_{tag}'] = preds[:, i]
    pred_df[f'prob_{tag}'] = tp[:, i].round(3)
pred_df.to_csv('saved_data/test_predictions.csv', index=False)
print("Saved: saved_data/test_predictions.csv")


## Block 13 — Human Review Export
Exports three batches of complaints to `human_review/complaints_for_review.csv`:

| Category | Why exported | How many |
|---|---|---|
| **Uncertain** | High entropy — model unsure, needs human label | ~33% of review budget |
| **Rare-tag predicted** | Verify model is correct on rare tag predictions | ~33% |
| **High-confidence** | Spot-check for systematic overconfidence | ~33% |

**How to use the CSV:**
1. Open `human_review/complaints_for_review.csv`
2. Read `summary` and `model_predicted_tags`
3. Fill `human_corrected_tags` with correct tier_2 tags (comma-separated)
4. Run Block 14 to merge corrections back

In [ ]:
def prediction_entropy(probs):
    p = np.clip(probs, 1e-9, 1-1e-9)
    return -(p * np.log(p) + (1-p) * np.log(1-p)).mean(axis=1)

entropy  = prediction_entropy(tp)
n_review = min(100, max(20, int(len(df_te) * 0.20)))
n_each   = max(5, n_review // 3)
base_cols = ['case_reference', 'summary', 'Service', 'tags_parsed']

def make_pred_tags(idx_list):
    return [', '.join(sorted([valid_tags[j] for j in np.where(preds[i]==1)[0]])) for i in idx_list]

# A) Most uncertain
uncertain_idx = np.argsort(entropy)[-n_each:]
df_unc = df_te.iloc[uncertain_idx][base_cols].copy()
df_unc['model_predicted_tags'] = make_pred_tags(uncertain_idx)
df_unc['model_confidence']     = entropy[uncertain_idx].round(3)
df_unc['review_reason']        = 'Uncertain — low model confidence'
df_unc['human_corrected_tags'] = ''

# B) Rare-tag predictions
rare_tag_idx_set = set(np.where(tag_freq < RARE_FREQ)[0])
rare_pred_rows   = [i for i in range(len(df_te))
                    if any(preds[i,j]==1 for j in rare_tag_idx_set)][:n_each]
df_rare = df_te.iloc[rare_pred_rows][base_cols].copy()
df_rare['model_predicted_tags'] = make_pred_tags(rare_pred_rows)
df_rare['model_confidence']     = [round(float(tp[i].max()), 3) for i in rare_pred_rows]
df_rare['review_reason']        = 'Rare tag predicted — please verify'
df_rare['human_corrected_tags'] = ''

# C) High confidence spot-check
high_conf_idx = np.argsort(-tp.max(axis=1))[:n_each]
df_hc = df_te.iloc[high_conf_idx][base_cols].copy()
df_hc['model_predicted_tags'] = make_pred_tags(high_conf_idx)
df_hc['model_confidence']     = tp[high_conf_idx].max(axis=1).round(3)
df_hc['review_reason']        = 'High confidence — spot-check for overconfidence'
df_hc['human_corrected_tags'] = ''

review_df = pd.concat([df_unc, df_rare, df_hc], ignore_index=True)              .drop_duplicates(subset=['case_reference'])
review_df.to_csv('human_review/complaints_for_review.csv', index=False)

print(f"Human review file exported: human_review/complaints_for_review.csv")
print(f"  Total rows:  {len(review_df)}")
print(f"  Uncertain:   {len(df_unc)}")
print(f"  Rare-tag:    {len(df_rare)}")
print(f"  High-conf:   {len(df_hc)}")
print()
print("NEXT STEPS:")
print("  1. Open human_review/complaints_for_review.csv")
print("  2. Fill 'human_corrected_tags' column (comma-separated tier_2 tags)")
print("  3. Run Block 14 to merge corrections and retrain")


## Block 14 — Feedback Loop (Run After Human Review)
Merges your human corrections back into the training set, then guides you to retrain.

In [ ]:
def apply_human_feedback(corrected_csv_path='human_review/complaints_for_review.csv'):
    """
    Run this AFTER filling 'human_corrected_tags' in the review CSV.
    Merges corrections into training data and saves updated set.

    Then go back to Block 2, set:
        DATA_FILE = 'saved_data/train_with_feedback.csv'
    and re-run from Block 2 onwards.
    """
    if not Path(corrected_csv_path).exists():
        print(f"File not found: {corrected_csv_path}")
        print("Complete Block 13 first"); return

    corrected = pd.read_csv(corrected_csv_path)
    corrected = corrected[corrected['human_corrected_tags'].notna() &
                          (corrected['human_corrected_tags'].str.strip() != '')]

    if corrected.empty:
        print("No corrections found — fill the 'human_corrected_tags' column first"); return

    print(f"Found {len(corrected)} human corrections — merging...")
    corrected['tags_parsed'] = corrected['human_corrected_tags'].apply(
        lambda x: [t.strip() for t in str(x).split(',') if t.strip()])

    df_feedback = pd.read_csv('saved_data/train_split.csv')
    df_feedback['tags_parsed'] = df_feedback['tags_parsed'].apply(parse_tags)
    count = 0

    for _, row in corrected.iterrows():
        match = df_feedback['case_reference'] == row['case_reference']
        if match.any():
            df_feedback.loc[match, 'tags_parsed'] = str(row['tags_parsed']); count += 1
        else:
            df_feedback = pd.concat([df_feedback, pd.DataFrame([row])], ignore_index=True)
            count += 1

    df_feedback.to_csv('saved_data/train_with_feedback.csv', index=False)
    print(f"Applied {count} corrections")
    print("Saved: saved_data/train_with_feedback.csv")
    print()
    print("TO RETRAIN WITH CORRECTIONS:")
    print("  1. Go to Block 2")
    print("  2. Set DATA_FILE = 'saved_data/train_with_feedback.csv'")
    print("  3. Re-run all blocks from Block 2 onwards")

apply_human_feedback()


## Block 15 — Inference on New Complaints
Use the trained model to predict tier_2 tags on any new complaint row.

In [ ]:
def predict_tags(row_dict, top_k=None):
    """
    Predict tier_2 tags for a single complaint.

    Parameters:
        row_dict : dict — keys: summary, Service,
                          tier2_consolidated_description (optional),
                          keyword_group_parsed (optional list)
        top_k    : int — if no tag passes threshold, return top_k by probability

    Returns:
        tags : list of predicted tag strings
        conf : dict of {tag: probability}
    """
    row  = pd.Series(row_dict)
    text = build_text(row, is_training=False)
    enc  = tokenizer(text, max_length=MAX_LEN, padding='max_length',
                     truncation=True, return_tensors='pt')
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(
            model(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE))
        ).cpu().numpy()[0]

    idx  = np.where(probs >= thresholds)[0]
    if top_k and len(idx) == 0:
        idx = np.argsort(probs)[-top_k:]

    tags = sorted([valid_tags[i] for i in idx])
    conf = {valid_tags[i]: round(float(probs[i]), 3)
            for i in sorted(idx, key=lambda x: -probs[x])}
    return tags, conf


# Example usage
example_complaint = {
    'Service':                        'Vehicle loan or lease',
    'summary':                        'Account breach on my vehicle loan. I disputed immediately but their investigation was inadequate.',
    'tier2_consolidated_description': 'Account breach Fraudulent charges',
    'keyword_group_parsed':           ['Fraud and security', 'Consumer protection'],
}

tags, conf = predict_tags(example_complaint, top_k=3)
print(f"Predicted tags: {tags}")
print()
print("Confidence scores:")
for tag, score in conf.items():
    bar = '#' * int(score * 20)
    print(f"  {tag:<40} {score:.3f}  {bar}")

print()
print("Batch prediction on a new CSV:")
print("  new_df = pd.read_csv('new_complaints.csv')")
print("  results = new_df.apply(lambda r: predict_tags(r.to_dict()), axis=1)")
print("  new_df['predicted_tags'], new_df['confidence'] = zip(*results)")


## Block 16 — Rich Prediction Output: Top-10 Keywords, Confidence & Backtracking

For every complaint in the test set this block produces:
- **Top 10 predicted tier_2 tags** with % confidence bars
- **Actual tags** side-by-side
- **Match / Miss / False-Alarm** classification per tag
- **Backtracking explanation** for mismatches — which input tokens most influenced the wrong prediction (via attention weights on [CLS])
- Full results saved to `saved_data/rich_predictions.csv`
- Human-readable HTML report saved to `saved_data/prediction_report.html`


In [ ]:
import pandas as pd
import numpy as np
import torch
import json
from pathlib import Path

# ── Reload everything if running this block standalone ────────
if 'model' not in dir():
    from transformers import AutoTokenizer, AutoModel
    import torch.nn as nn
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model.load_state_dict(torch.load('checkpoints/best_model.pt', map_location=DEVICE))
    thresholds = np.load('saved_data/tag_thresholds.npy')
    with open('saved_data/valid_tags.json') as f:
        valid_tags = json.load(f)
    df_te  = pd.read_csv('saved_data/test_split.csv')
    y_te   = np.load('saved_data/y_te.npy')
    NUM_LABELS = len(valid_tags)

model.eval()
TOP_K        = 10     # top predicted tags to show
N_TOKENS     = 8      # top input tokens to surface in backtrack
N_SHOW       = len(df_te)   # set lower (e.g. 20) to preview faster

# ── HELPER: get attention rollout for backtracking ─────────────
def get_top_tokens(text, model, tokenizer, tag_idx, device, n=N_TOKENS):
    """
    Uses the [CLS] attention from the last layer to find which
    input tokens most influenced the prediction for tag_idx.
    Returns list of (token, weight_pct) tuples.
    """
    enc = tokenizer(text, max_length=MAX_LEN, padding='max_length',
                    truncation=True, return_tensors='pt')
    input_ids      = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)

    with torch.no_grad():
        out = model.bert(input_ids=input_ids,
                         attention_mask=attention_mask,
                         output_attentions=True)

    # Average attention from last layer, from [CLS] token to all others
    last_layer_attn = out.attentions[-1]          # (1, heads, seq, seq)
    cls_attn        = last_layer_attn[0, :, 0, :] # (heads, seq)
    avg_attn        = cls_attn.mean(dim=0).cpu().numpy()  # (seq,)

    tokens   = tokenizer.convert_ids_to_tokens(input_ids[0].cpu().numpy())
    pad_id   = tokenizer.pad_token_id
    real_len = int(attention_mask[0].sum().item())

    # Remove [CLS], [SEP], [PAD]
    clean = [(tok, avg_attn[i])
             for i, tok in enumerate(tokens[:real_len])
             if tok not in ('[CLS]','[SEP]','[PAD]','<s>','</s>')
             and not tok.startswith('##')]

    if not clean:
        return []

    toks, weights = zip(*clean)
    weights = np.array(weights)
    weights = weights / (weights.sum() + 1e-9) * 100
    top_idx = np.argsort(weights)[-n:][::-1]
    return [(toks[i], round(float(weights[i]), 1)) for i in top_idx]


# ── MAIN LOOP ──────────────────────────────────────────────────
print(f"Generating rich predictions for {N_SHOW} test complaints...\n")

records  = []
html_rows = []

for row_i in range(N_SHOW):
    row      = df_te.iloc[row_i]
    true_lbl = y_te[row_i]

    # Build input text (same function as training)
    text = build_text(row, is_training=False)

    enc = tokenizer(text, max_length=MAX_LEN, padding='max_length',
                    truncation=True, return_tensors='pt')
    with torch.no_grad():
        logits = model(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE))
        probs  = torch.sigmoid(logits).cpu().numpy()[0]

    # Top-10 predicted tags by probability (regardless of threshold)
    top10_idx   = np.argsort(probs)[-TOP_K:][::-1]
    top10_tags  = [(valid_tags[i], round(float(probs[i])*100, 1)) for i in top10_idx]

    # Tags that pass threshold (actual predictions)
    pred_idx    = np.where(probs >= thresholds)[0]
    pred_tags   = set(valid_tags[i] for i in pred_idx)
    actual_tags = set(valid_tags[i] for i in np.where(true_lbl == 1)[0])

    correct      = pred_tags & actual_tags       # true positives
    missed       = actual_tags - pred_tags        # false negatives
    false_alarms = pred_tags - actual_tags        # false positives
    is_perfect   = (pred_tags == actual_tags)

    # Backtrack mismatches — top influencing tokens for each mismatch
    backtrack = {}
    for tag in (missed | false_alarms):
        tag_idx = list(valid_tags).index(tag)
        top_toks = get_top_tokens(text, model, tokenizer, tag_idx, DEVICE)
        direction = "MISSED" if tag in missed else "FALSE ALARM"
        conf_pct  = round(float(probs[tag_idx]) * 100, 1)
        backtrack[tag] = {
            "direction":   direction,
            "confidence":  conf_pct,
            "top_tokens":  top_toks
        }

    # ── Console output (first 10 complaints always shown) ──────
    if row_i < 10:
        ref = str(row.get('case_reference', row_i))
        print("=" * 70)
        print(f"Complaint #{row_i+1}  |  Ref: {ref}")
        print(f"Summary: {str(row.get('summary',''))[:120]}...")
        print()
        print(f"{'TAG':<42} {'CONF%':>6}  {'STATUS'}")
        print("-" * 70)
        for tag, pct in top10_tags:
            passed    = tag in pred_tags
            is_actual = tag in actual_tags
            if is_actual and passed:   status = "CORRECT"
            elif is_actual and not passed: status = "MISSED  <--"
            elif passed and not is_actual: status = "FALSE ALARM <--"
            else:                      status = "not predicted"
            bar = "#" * int(pct / 5)
            print(f"  {tag:<40} {pct:>5.1f}%  {bar}  {status}")

        print()
        if actual_tags:
            print(f"  Actual tags:  {sorted(actual_tags)}")
        if missed:
            print(f"  Missed:       {sorted(missed)}")
        if false_alarms:
            print(f"  False alarms: {sorted(false_alarms)}")

        if backtrack:
            print()
            print("  BACKTRACKING (mismatch analysis):")
            for tag, info in backtrack.items():
                print(f"    [{info['direction']}] '{tag}'  confidence={info['confidence']}%")
                if info['top_tokens']:
                    tok_str = "  |  ".join(
                        f"{t}({w}%)" for t,w in info['top_tokens'][:5])
                    print(f"      Top tokens driving this: {tok_str}")
        print()

    # ── Store record for CSV ───────────────────────────────────
    rec = {
        'case_reference':  row.get('case_reference', row_i),
        'summary_snippet': str(row.get('summary',''))[:200],
        'Service':         row.get('Service',''),
        'actual_tags':     ', '.join(sorted(actual_tags)),
        'predicted_tags':  ', '.join(sorted(pred_tags)),
        'correct_tags':    ', '.join(sorted(correct)),
        'missed_tags':     ', '.join(sorted(missed)),
        'false_alarm_tags':', '.join(sorted(false_alarms)),
        'is_perfect_match': is_perfect,
        'n_actual':        len(actual_tags),
        'n_predicted':     len(pred_tags),
        'n_correct':       len(correct),
    }
    for tag, pct in top10_tags:
        rec[f'top10__{tag.replace(" ","_")}_conf%'] = pct
    for tag, info in backtrack.items():
        key = tag.replace(" ","_")
        rec[f'backtrack__{key}__direction']   = info['direction']
        rec[f'backtrack__{key}__confidence%'] = info['confidence']
        rec[f'backtrack__{key}__top_tokens']  = str(info['top_tokens'][:5])
    records.append(rec)

    # ── HTML row ───────────────────────────────────────────────
    top10_html = "".join([
        f'<div class="tag-row"><span class="tag-name">{t}</span>'
        f'<div class="bar-wrap"><div class="bar {\'match\' if t in actual_tags and t in pred_tags else \'miss\' if t in actual_tags else \'fa\' if t in pred_tags else \'na\'}"'
        f' style="width:{min(p,100):.0f}%"></div></div>'
        f'<span class="pct">{p:.1f}%</span></div>'
        for t, p in top10_tags
    ])
    bt_html = ""
    for tag, info in backtrack.items():
        toks = " | ".join(f"{t} ({w}%)" for t,w in info['top_tokens'][:5])
        cls  = "missed" if info['direction'] == "MISSED" else "fa"
        bt_html += (f'<div class="bt {cls}"><b>[{info["direction"]}]</b> '{tag}' '
                    f'conf={info["confidence"]}% &rarr; <span class="toks">{toks}</span></div>')

    ref_val = str(row.get('case_reference', row_i))
    svc_val = str(row.get('Service', ''))
    summ_val = str(row.get('summary',''))[:200].replace("<","&lt;")

    html_rows.append(f"""
    <tr>
      <td class="ref">{ref_val}<br><small>{svc_val}</small></td>
      <td class="summ">{summ_val}...</td>
      <td class="tags actual">{" ".join(f'<span class="pill">{t}</span>' for t in sorted(actual_tags))}</td>
      <td class="top10">{top10_html}</td>
      <td class="bt-cell">{bt_html if bt_html else '<span class="ok">All correct</span>'}</td>
    </tr>""")

# ── Save CSV ──────────────────────────────────────────────────
rich_df = pd.DataFrame(records)
rich_df.to_csv('saved_data/rich_predictions.csv', index=False)
print(f"\nSaved: saved_data/rich_predictions.csv  ({len(rich_df)} rows)")

# ── Summary stats ─────────────────────────────────────────────
perfect_pct = rich_df['is_perfect_match'].mean()*100
print(f"\nPREDICTION SUMMARY ({N_SHOW} complaints):")
print(f"  Perfect matches:    {rich_df['is_perfect_match'].sum()} / {N_SHOW}  ({perfect_pct:.1f}%)")
print(f"  Avg tags predicted: {rich_df['n_predicted'].mean():.2f}")
print(f"  Avg tags actual:    {rich_df['n_actual'].mean():.2f}")
print(f"  Avg correct:        {rich_df['n_correct'].mean():.2f}")

# ── Save HTML report ──────────────────────────────────────────
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Complaint Tagger — Prediction Report</title>
<style>
  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
          font-size: 13px; background:#f5f5f2; color:#222; margin:0; padding:16px; }}
  h1   {{ font-size:20px; color:#01696f; margin-bottom:4px; }}
  p.sub {{ color:#666; margin-top:0; font-size:12px; }}
  table {{ border-collapse:collapse; width:100%; background:#fff;
           box-shadow:0 1px 4px rgba(0,0,0,.08); border-radius:8px; overflow:hidden; }}
  th   {{ background:#01696f; color:#fff; padding:10px 8px; text-align:left; font-size:12px; }}
  td   {{ padding:10px 8px; border-bottom:1px solid #eee; vertical-align:top; }}
  tr:hover td {{ background:#f0fafa; }}
  .ref  {{ font-weight:600; color:#01696f; white-space:nowrap; min-width:100px; }}
  .summ {{ max-width:260px; color:#444; font-size:12px; }}
  .actual {{ max-width:160px; }}
  .pill {{ display:inline-block; background:#cedcd8; color:#0c4e54;
           border-radius:99px; padding:2px 8px; font-size:11px; margin:2px; }}
  .tag-row {{ display:flex; align-items:center; gap:6px; margin:3px 0; }}
  .tag-name {{ min-width:180px; font-size:11px; color:#333; }}
  .bar-wrap {{ width:100px; height:10px; background:#eee; border-radius:5px; overflow:hidden; }}
  .bar {{ height:100%; border-radius:5px; }}
  .bar.match {{ background:#437a22; }}
  .bar.miss  {{ background:#e07b39; }}
  .bar.fa    {{ background:#a12c7b; }}
  .bar.na    {{ background:#bbb; }}
  .pct  {{ font-size:11px; color:#555; min-width:40px; }}
  .bt   {{ font-size:11px; margin:4px 0; padding:4px 8px; border-radius:4px; }}
  .bt.missed {{ background:#fff3ec; border-left:3px solid #e07b39; }}
  .bt.fa     {{ background:#f9ecf5; border-left:3px solid #a12c7b; }}
  .toks {{ color:#555; font-style:italic; }}
  .ok   {{ color:#437a22; font-size:11px; }}
  .legend {{ display:flex; gap:16px; margin:12px 0 16px; font-size:12px; }}
  .legend span {{ display:flex; align-items:center; gap:5px; }}
  .dot {{ width:12px; height:12px; border-radius:50%; display:inline-block; }}
</style>
</head>
<body>
<h1>Complaint Tagger — Prediction Report</h1>
<p class="sub">
  {N_SHOW} complaints | {len(valid_tags)} tags |
  Perfect matches: {rich_df["is_perfect_match"].sum()} ({perfect_pct:.1f}%)
</p>
<div class="legend">
  <span><div class="dot" style="background:#437a22"></div> Correct prediction</span>
  <span><div class="dot" style="background:#e07b39"></div> Missed (actual tag not predicted)</span>
  <span><div class="dot" style="background:#a12c7b"></div> False alarm (predicted, not actual)</span>
  <span><div class="dot" style="background:#bbb"></div> Not predicted</span>
</div>
<table>
<thead>
  <tr>
    <th>Reference</th>
    <th>Summary</th>
    <th>Actual Tags</th>
    <th>Top 10 Predictions + Confidence</th>
    <th>Mismatch Backtracking</th>
  </tr>
</thead>
<tbody>
{"".join(html_rows)}
</tbody>
</table>
</body>
</html>"""

with open('saved_data/prediction_report.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("Saved: saved_data/prediction_report.html  (open in any browser)")
print()
print("LEGEND:")
print("  GREEN bar  = correct prediction (tag in both predicted and actual)")
print("  ORANGE bar = missed tag (actual but not predicted)")
print("  PURPLE bar = false alarm (predicted but not actual)")
print("  GREY bar   = in top-10 by probability but below threshold")
print()
print("BACKTRACKING:")
print("  Shows which input tokens most influenced the mismatch via")
print("  attention weights from the [CLS] token in the last BERT layer.")


## Block 17 — Standalone Output Analysis & Testing Suite

**Run this block independently at any time** — it reads from `saved_data/` files only.
No model reloading needed. Takes the saved prediction outputs and generates:

1. **Full model performance report** — Macro/Micro/Samples F1, per-tag breakdown
2. **Tag-level analysis table** — confidence, TP/FP/FN counts, threshold used
3. **Confusion heatmap** — which tags get confused with each other most
4. **Confidence calibration check** — is high confidence actually correct?
5. **Backtracking summary** — which input tokens drive most errors
6. **Failure mode analysis** — systematic patterns in wrong predictions
7. **HTML dashboard** — everything in one file, open in browser

> **Inputs needed:** `saved_data/rich_predictions.csv`, `saved_data/tag_info.csv`, `saved_data/test_split.csv`
> All produced by Blocks 12–16.


In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import f1_score, precision_score, recall_score

Path("saved_data").mkdir(exist_ok=True)

# ════════════════════════════════════════════════════════════════
# LOAD ALL SAVED OUTPUTS
# ════════════════════════════════════════════════════════════════
print("Loading saved outputs...")
rich_df  = pd.read_csv("saved_data/rich_predictions.csv")
tag_info = pd.read_csv("saved_data/tag_info.csv")
df_te    = pd.read_csv("saved_data/test_split.csv")

with open("saved_data/valid_tags.json") as f:
    valid_tags = json.load(f)

thresholds = np.load("saved_data/tag_thresholds.npy")
y_te       = np.load("saved_data/y_te.npy")

# Rebuild pred/prob matrices from rich_predictions.csv
conf_cols  = [c for c in rich_df.columns if c.startswith("top10__") and c.endswith("_conf%")]
tag_map    = {c: c.replace("top10__","").replace("_conf%","").replace("_"," ") for c in conf_cols}

# Reconstruct full prob + pred arrays
prob_matrix = np.zeros((len(rich_df), len(valid_tags)))
pred_matrix = np.zeros((len(rich_df), len(valid_tags)), dtype=int)

for col, tag_raw in tag_map.items():
    # match back to valid_tags (handle underscore vs space)
    matches = [i for i,t in enumerate(valid_tags) if t.replace(" ","_") == tag_raw.replace(" ","_")]
    if matches:
        idx = matches[0]
        prob_matrix[:, idx] = rich_df[col].fillna(0).values / 100.0

for i, row in rich_df.iterrows():
    pred_tags_str = str(row.get("predicted_tags",""))
    pred_tags_list = [t.strip() for t in pred_tags_str.split(",") if t.strip()]
    for t in pred_tags_list:
        if t in valid_tags:
            pred_matrix[i, valid_tags.index(t)] = 1

# True label matrix
true_matrix = (y_te >= 0.5).astype(int)[:len(rich_df)]

print(f"  Loaded: {len(rich_df)} complaints | {len(valid_tags)} tags")
print(f"  Pred matrix: {pred_matrix.shape} | True matrix: {true_matrix.shape}")

# ════════════════════════════════════════════════════════════════
# 1. OVERALL PERFORMANCE METRICS
# ════════════════════════════════════════════════════════════════
print()
print("=" * 60)
print("  OVERALL PERFORMANCE")
print("=" * 60)

macro_f1  = f1_score(true_matrix, pred_matrix, average="macro",   zero_division=0)
micro_f1  = f1_score(true_matrix, pred_matrix, average="micro",   zero_division=0)
samp_f1   = f1_score(true_matrix, pred_matrix, average="samples", zero_division=0)
macro_p   = precision_score(true_matrix, pred_matrix, average="macro",  zero_division=0)
macro_r   = recall_score(true_matrix,    pred_matrix, average="macro",  zero_division=0)

perfect_n = int(rich_df["is_perfect_match"].sum())
perfect_pct = rich_df["is_perfect_match"].mean() * 100

print(f"  Macro  F1:        {macro_f1:.4f}   <- main metric (equal weight all tags)")
print(f"  Micro  F1:        {micro_f1:.4f}   <- weighted by tag frequency")
print(f"  Samples F1:       {samp_f1:.4f}   <- per-complaint average")
print(f"  Macro Precision:  {macro_p:.4f}")
print(f"  Macro Recall:     {macro_r:.4f}")
print(f"  Perfect matches:  {perfect_n}/{len(rich_df)}  ({perfect_pct:.1f}%)")
print()

# ════════════════════════════════════════════════════════════════
# 2. PER-TAG BREAKDOWN TABLE
# ════════════════════════════════════════════════════════════════
print("=" * 60)
print("  PER-TAG BREAKDOWN")
print("=" * 60)

tag_rows = []
for i, tag in enumerate(valid_tags):
    y_true_t = true_matrix[:, i]
    y_pred_t = pred_matrix[:, i]
    tp = int((y_true_t * y_pred_t).sum())
    fp = int(((1-y_true_t) * y_pred_t).sum())
    fn = int((y_true_t * (1-y_pred_t)).sum())
    tn = int(((1-y_true_t) * (1-y_pred_t)).sum())
    prec   = tp / (tp+fp) if (tp+fp) > 0 else 0.0
    rec    = tp / (tp+fn) if (tp+fn) > 0 else 0.0
    f1_t   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
    support = int(y_true_t.sum())

    # avg confidence when predicted
    pred_conf_vals = prob_matrix[:, i][pred_matrix[:, i]==1] * 100
    avg_conf = round(pred_conf_vals.mean(), 1) if len(pred_conf_vals) else 0.0

    tinfo = tag_info[tag_info["tag"] == tag]
    thresh = float(tinfo["threshold"].values[0]) if len(tinfo) else 0.5
    category = str(tinfo["category"].values[0]) if len(tinfo) else "normal"

    tag_rows.append({
        "tag":        tag,
        "category":   category,
        "support":    support,
        "TP":         tp, "FP": fp, "FN": fn,
        "precision":  round(prec, 3),
        "recall":     round(rec, 3),
        "f1":         round(f1_t, 3),
        "avg_conf%":  avg_conf,
        "threshold":  round(thresh, 2),
    })

tag_df = pd.DataFrame(tag_rows).sort_values("f1", ascending=False)
tag_df.to_csv("saved_data/per_tag_results.csv", index=False)

# Print nicely
print(f"{'TAG':<38} {'CAT':<9} {'SUP':>4} {'F1':>6} {'PREC':>6} {'REC':>6} {'TP':>4} {'FP':>4} {'FN':>4} {'CONF%':>6} {'THR':>5}")
print("-" * 105)
for _, r in tag_df.iterrows():
    print(f"  {r['tag']:<36} {r['category']:<9} {r['support']:>4} "
          f"{r['f1']:>6.3f} {r['precision']:>6.3f} {r['recall']:>6.3f} "
          f"{r['TP']:>4} {r['FP']:>4} {r['FN']:>4} "
          f"{r['avg_conf%']:>5.1f}% {r['threshold']:>5.2f}")

print(f"\nSaved: saved_data/per_tag_results.csv")

# ════════════════════════════════════════════════════════════════
# 3. CONFIDENCE CALIBRATION CHECK
# How accurate is the model when it is confident vs uncertain?
# ════════════════════════════════════════════════════════════════
print()
print("=" * 60)
print("  CONFIDENCE CALIBRATION")
print("=" * 60)

bins = [(0,30,"< 30%"),(30,50,"30-50%"),(50,70,"50-70%"),(70,90,"70-90%"),(90,101,"90%+")]
print(f"  {'Confidence band':<14} {'Predictions':>12} {'Correct':>9} {'Accuracy':>10}")
print("  " + "-" * 50)

calib_rows = []
for lo, hi, label in bins:
    mask = (prob_matrix * 100 >= lo) & (prob_matrix * 100 < hi) & (pred_matrix == 1)
    n_preds = int(mask.sum())
    if n_preds == 0:
        print(f"  {label:<14} {'0':>12}")
        continue
    correct = int((mask & (true_matrix == 1)).sum())
    acc = correct / n_preds * 100
    bar = "#" * int(acc / 5)
    print(f"  {label:<14} {n_preds:>12,} {correct:>9,} {acc:>9.1f}%  {bar}")
    calib_rows.append({"band": label, "predictions": n_preds,
                        "correct": correct, "accuracy_%": round(acc, 1)})

pd.DataFrame(calib_rows).to_csv("saved_data/calibration.csv", index=False)
print(f"\nSaved: saved_data/calibration.csv")

# ════════════════════════════════════════════════════════════════
# 4. FAILURE MODE ANALYSIS
# What are the most common error patterns?
# ════════════════════════════════════════════════════════════════
print()
print("=" * 60)
print("  FAILURE MODE ANALYSIS")
print("=" * 60)

# Most missed tags
fn_counts = {}
for _, r in rich_df.iterrows():
    for t in str(r.get("missed_tags","")).split(","):
        t = t.strip()
        if t: fn_counts[t] = fn_counts.get(t, 0) + 1

# Most false alarmed tags
fp_counts = {}
for _, r in rich_df.iterrows():
    for t in str(r.get("false_alarm_tags","")).split(","):
        t = t.strip()
        if t: fp_counts[t] = fp_counts.get(t, 0) + 1

top_missed = sorted(fn_counts.items(), key=lambda x: -x[1])[:10]
top_false  = sorted(fp_counts.items(), key=lambda x: -x[1])[:10]

print(f"\n  Top 10 most MISSED tags (false negatives):")
print(f"  {'Tag':<42} {'Count':>6}")
print("  " + "-" * 50)
for tag, cnt in top_missed:
    bar = "#" * cnt
    print(f"  {tag:<42} {cnt:>6}  {bar}")

print(f"\n  Top 10 most FALSE ALARMED tags (false positives):")
print(f"  {'Tag':<42} {'Count':>6}")
print("  " + "-" * 50)
for tag, cnt in top_false:
    bar = "#" * cnt
    print(f"  {tag:<42} {cnt:>6}  {bar}")

# Most confused pairs (tag A predicted when tag B was actual)
conf_pairs = {}
for _, r in rich_df.iterrows():
    missed_list = [t.strip() for t in str(r.get("missed_tags","")).split(",") if t.strip()]
    fa_list     = [t.strip() for t in str(r.get("false_alarm_tags","")).split(",") if t.strip()]
    for m in missed_list:
        for f in fa_list:
            if m != f:
                pair = tuple(sorted([m, f]))
                conf_pairs[pair] = conf_pairs.get(pair, 0) + 1

top_confused = sorted(conf_pairs.items(), key=lambda x: -x[1])[:10]
print(f"\n  Top confused tag PAIRS (model predicted B when A was correct):")
print(f"  {'Actual → Predicted':<60} {'Count':>6}")
print("  " + "-" * 68)
for (a, b), cnt in top_confused:
    print(f"  {a}  <->  {b}  |  confused {cnt}x")

failure_df = pd.DataFrame({
    "most_missed_tag":   [t for t,_ in top_missed],
    "missed_count":      [c for _,c in top_missed],
    "most_false_alarm":  [t for t,_ in top_false] + [""]*max(0,len(top_missed)-len(top_false)),
    "false_alarm_count": [c for _,c in top_false] + [0]*max(0,len(top_missed)-len(top_false)),
})
failure_df.to_csv("saved_data/failure_modes.csv", index=False)
print(f"\nSaved: saved_data/failure_modes.csv")

# ════════════════════════════════════════════════════════════════
# 5. BACKTRACKING TOKEN SUMMARY
# Which tokens appear most in mismatch explanations?
# ════════════════════════════════════════════════════════════════
print()
print("=" * 60)
print("  BACKTRACKING TOKEN SUMMARY")
print("=" * 60)

bt_cols_missed = [c for c in rich_df.columns
                  if "backtrack__" in c and "__top_tokens" in c
                  and "_MISSED_" not in c]  # all backtrack token cols

token_freq = {}
for col in bt_cols_missed:
    for val in rich_df[col].dropna():
        try:
            toks = eval(val)  # stored as list of (token, weight) tuples
            for tok, w in toks:
                tok = tok.lower().strip(".,!?#")
                if len(tok) > 2:
                    token_freq[tok] = token_freq.get(tok, 0) + 1
        except:
            pass

top_tokens = sorted(token_freq.items(), key=lambda x: -x[1])[:20]
if top_tokens:
    print(f"  Top 20 tokens driving mismatch predictions:")
    print(f"  {'Token':<25} {'Appearances':>12}")
    print("  " + "-" * 40)
    for tok, cnt in top_tokens:
        bar = "#" * min(cnt, 30)
        print(f"  {tok:<25} {cnt:>12}  {bar}")
else:
    print("  No backtracking data found — run Block 16 first with output_attentions=True")

# ════════════════════════════════════════════════════════════════
# 6. SAVE FULL HTML DASHBOARD
# ════════════════════════════════════════════════════════════════
print()
print("=" * 60)
print("  GENERATING HTML DASHBOARD")
print("=" * 60)

# Tag performance table rows
tag_table_rows = ""
for _, r in tag_df.iterrows():
    f1_val  = r["f1"]
    f1_col  = "#437a22" if f1_val >= 0.7 else "#d19900" if f1_val >= 0.45 else "#a12c7b"
    cat_col = "#01696f" if r["category"]=="dominant" else "#a12c7b" if r["category"]=="rare" else "#555"
    tag_table_rows += f"""
    <tr>
      <td>{r["tag"]}</td>
      <td style="color:{cat_col};font-weight:600">{r["category"]}</td>
      <td>{r["support"]}</td>
      <td style="color:{f1_col};font-weight:700">{r["f1"]:.3f}</td>
      <td>{r["precision"]:.3f}</td>
      <td>{r["recall"]:.3f}</td>
      <td>{r["TP"]}</td>
      <td style="color:#a12c7b">{r["FP"]}</td>
      <td style="color:#e07b39">{r["FN"]}</td>
      <td>{r["avg_conf%"]:.1f}%</td>
      <td>{r["threshold"]:.2f}</td>
    </tr>"""

# Calibration rows
calib_rows_html = ""
for row in calib_rows:
    acc = row["accuracy_%"]
    col = "#437a22" if acc >= 80 else "#d19900" if acc >= 60 else "#a12c7b"
    bar_w = int(acc)
    calib_rows_html += f"""
    <tr>
      <td>{row["band"]}</td>
      <td>{row["predictions"]:,}</td>
      <td>{row["correct"]:,}</td>
      <td>
        <div style="display:flex;align-items:center;gap:8px">
          <div style="width:{bar_w}px;height:12px;background:{col};border-radius:3px"></div>
          <span style="color:{col};font-weight:700">{acc}%</span>
        </div>
      </td>
    </tr>"""

# Failure mode rows
fail_rows_html = ""
for tag, cnt in top_missed[:8]:
    fail_rows_html += f'<tr><td>{tag}</td><td style="color:#e07b39">{cnt}</td><td>Missed (FN)</td></tr>'
for tag, cnt in top_false[:8]:
    fail_rows_html += f'<tr><td>{tag}</td><td style="color:#a12c7b">{cnt}</td><td>False alarm (FP)</td></tr>'

# Confused pairs rows
conf_rows_html = ""
for (a, b), cnt in top_confused[:8]:
    conf_rows_html += f"<tr><td>{a}</td><td>{b}</td><td>{cnt}</td></tr>"

html_dash = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Complaint Tagger — Analysis Dashboard</title>
<style>
  * {{ box-sizing:border-box; margin:0; padding:0; }}
  body {{ font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;
          background:#f5f5f2; color:#222; font-size:13px; padding:20px; }}
  h1   {{ font-size:22px; color:#01696f; margin-bottom:4px; }}
  h2   {{ font-size:15px; color:#01696f; margin:28px 0 10px;
          padding-bottom:6px; border-bottom:2px solid #cedcd8; }}
  .meta {{ color:#777; font-size:12px; margin-bottom:20px; }}
  .kpi-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(160px,1fr));
               gap:12px; margin:16px 0 24px; }}
  .kpi {{ background:#fff; border-radius:8px; padding:16px;
          box-shadow:0 1px 4px rgba(0,0,0,.07); text-align:center; }}
  .kpi .val {{ font-size:28px; font-weight:700; color:#01696f; }}
  .kpi .lbl {{ font-size:11px; color:#888; margin-top:4px; }}
  table {{ border-collapse:collapse; width:100%; background:#fff;
           border-radius:8px; overflow:hidden;
           box-shadow:0 1px 4px rgba(0,0,0,.07); margin-bottom:24px; }}
  th {{ background:#01696f; color:#fff; padding:9px 10px;
        text-align:left; font-size:12px; white-space:nowrap; }}
  td {{ padding:8px 10px; border-bottom:1px solid #eee; vertical-align:middle; }}
  tr:hover td {{ background:#f0fafa; }}
  .section {{ background:#fff; border-radius:8px; padding:18px;
              box-shadow:0 1px 4px rgba(0,0,0,.07); margin-bottom:24px; }}
  .pill {{ display:inline-block; border-radius:99px; padding:2px 8px;
           font-size:11px; background:#cedcd8; color:#0c4e54; margin:2px; }}
</style>
</head>
<body>
<h1>Complaint Tagger — Analysis Dashboard</h1>
<p class="meta">{len(rich_df)} test complaints | {len(valid_tags)} tier_2 tags | Generated from saved_data/</p>

<div class="kpi-grid">
  <div class="kpi"><div class="val">{macro_f1:.3f}</div><div class="lbl">Macro F1</div></div>
  <div class="kpi"><div class="val">{micro_f1:.3f}</div><div class="lbl">Micro F1</div></div>
  <div class="kpi"><div class="val">{samp_f1:.3f}</div><div class="lbl">Samples F1</div></div>
  <div class="kpi"><div class="val">{macro_p:.3f}</div><div class="lbl">Macro Precision</div></div>
  <div class="kpi"><div class="val">{macro_r:.3f}</div><div class="lbl">Macro Recall</div></div>
  <div class="kpi"><div class="val">{perfect_pct:.1f}%</div><div class="lbl">Perfect Matches</div></div>
</div>

<h2>Per-Tag Performance (sorted by F1)</h2>
<table>
<thead>
  <tr><th>Tag</th><th>Category</th><th>Support</th><th>F1</th>
      <th>Precision</th><th>Recall</th><th>TP</th><th>FP</th><th>FN</th>
      <th>Avg Conf</th><th>Threshold</th></tr>
</thead>
<tbody>{tag_table_rows}</tbody>
</table>

<h2>Confidence Calibration</h2>
<p style="color:#666;font-size:12px;margin-bottom:10px">
  How accurate are predictions at each confidence level?
  High-confidence predictions should be more accurate than low-confidence ones.
</p>
<table>
<thead>
  <tr><th>Confidence Band</th><th>Predictions Made</th>
      <th>Correct (TP)</th><th>Accuracy</th></tr>
</thead>
<tbody>{calib_rows_html}</tbody>
</table>

<h2>Failure Mode Analysis</h2>
<table>
<thead><tr><th>Tag</th><th>Count</th><th>Error Type</th></tr></thead>
<tbody>{fail_rows_html}</tbody>
</table>

<h2>Most Confused Tag Pairs</h2>
<p style="color:#666;font-size:12px;margin-bottom:10px">
  Tag pairs most often swapped — candidates for annotation review or tag merging.
</p>
<table>
<thead><tr><th>Tag A</th><th>Tag B</th><th>Times Confused</th></tr></thead>
<tbody>{conf_rows_html}</tbody>
</table>

</body>
</html>"""

with open("saved_data/analysis_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html_dash)

print("  Saved: saved_data/analysis_dashboard.html  <- open in browser")
print()
print("=" * 60)
print("  ALL OUTPUT FILES SUMMARY")
print("=" * 60)
outputs = [
    ("saved_data/rich_predictions.csv",    "Per-complaint top-10 predictions + backtracking"),
    ("saved_data/per_tag_results.csv",     "Per-tag F1, precision, recall, TP/FP/FN"),
    ("saved_data/calibration.csv",         "Confidence calibration by band"),
    ("saved_data/failure_modes.csv",       "Most missed and false-alarmed tags"),
    ("saved_data/prediction_report.html",  "Per-complaint visual report (Block 16)"),
    ("saved_data/analysis_dashboard.html", "Full analysis dashboard (this block)"),
    ("saved_data/training_history.csv",    "Loss + F1 per epoch"),
    ("saved_data/test_predictions.csv",    "Raw predictions + probabilities per tag"),
]
for path, desc in outputs:
    exists = "OK" if Path(path).exists() else "MISSING — run earlier block"
    print(f"  [{exists:^7}]  {path}")
    print(f"             {desc}")
